# Lidocaine: 30 Hz burst vs ARC-EX, before and with lidocaine

**The question.** Lidocaine over the skin blocks cutaneous afferents. If a protocol recruits the
cord mainly through **dorsal-root afferents**, blocking the skin should leave it largely alone;
if a protocol drives **skin and superficial fibres** as well, blocking them should cost it more.
So: does lidocaine cost ARC-EX more than it costs 30 Hz, or the same?

**Two participants, one session each.**

| | | |
|---|---|---|
| **P04** | 17-07-2026, electrode 1 | before → lidocaine, both protocols |
| **P05** | 24-07-2026 (`testSCS`), electrode 2, **cathodic** | before → lidocaine, both protocols |

P04's electrode 3 and P05's anodic blocks exist in the session folders and are analysed in the
per-session notebooks; this comparison keeps one montage per participant so the two are read the
same way.

**Two things are measured, and they answer different halves of the question.**

1. **The threshold moves** — how much more current each protocol needs once the skin is blocked.
   This uses the thresholds you picked, nothing else.
2. **The response at a matched intensity shrinks** — with lidocaine as a % of the same
   muscle's response *before*, both read at the same mA, so the comparison is of the response
   and not of the current.

**The rule for intensity.** Every muscle is analysed **at its own motor threshold**, picked by
hand in section 2 — never at one mA for the whole arm, and never a step above. Within a protocol,
before and lidocaine are read at the **same** intensity (the before-threshold), or the bars would
compare current instead of lidocaine. 30 Hz and ARC-EX are never forced onto one intensity: they
sit ~70 mA apart and each carries its own thresholds.

**Colours, as everywhere else:** 30 Hz burst **grey**, ARC-EX **red**; the paler shade is
*before*, the solid one *with lidocaine*.

**Millivolts never travel between participants** — different electrodes, different skin, different
amplifier gain. Every comparison below is within a participant, and only the *changes* are read
across them.

In [ ]:
# run from the repo root so that src/, results/ and tSCS_CHUV_data/ resolve the same way from
# every notebook folder (VS Code starts the kernel in the notebook's own folder)
import os, sys
while not os.path.isdir("src") and os.getcwd() != "/":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from functions import (set_style, load_run, pretty, waterfall_overlay, thresholds_for, threshold_source,
                       condition_table, fig_vibration, fig_bars,
                       threshold_picker, save_threshold_csv, load_threshold_csv, mt_file,
                       mt_stamp, warn_if_stale,
                       fig_threshold_grid, fig_detection_grid, fig_group_summary, fig_group_panels,
                       snr_verdict, fig_noise_check)
set_style()
from functions.paper import motor_thresholds

## 1 · Config

In [ ]:
# ======== the recordings ========
D04 = "tSCS_CHUV_data/17-07-2026/P04tscsHealthy/"    # electrode 1
D06 = "tSCS_CHUV_data/28-08-2026/"                   # P06, 28-09-2026, electrode 2, CATHODIC
D05 = "tSCS_CHUV_data/24-07-2026/testSCS/"           # electrode 2, CATHODIC only:
#   that session also holds the anodic blocks (Burst 102721 / 114055, Modulated 103849 /
#   114730). They are the polarity comparison, in P05_cathodic_vs_anodic.ipynb - mixing
#   one in here would compare lidocaine with polarity.

RUN = {   # (subject, condition) -> csv
  ("P04", "30 Hz · before"):    D04 + "Burst_autosave_20260717_150152_655ms.csv",       # 15:01
  ("P04", "30 Hz · lidocaine"): D04 + "Burst_autosave_20260717_161911_499ms.csv",       # 16:19
  ("P04", "ARC-EX · before"):   D04 + "Modulated_autosave_20260717_150657_684ms.csv",   # 15:06
  ("P04", "ARC-EX · lidocaine"):D04 + "Modulated_autosave_20260717_162347_270ms.csv",   # 16:23
  ("P05", "30 Hz · before"):    D05 + "Burst_autosave_20260724_102443_670ms.csv",       # 10:24
  ("P05", "30 Hz · lidocaine"): D05 + "Burst_autosave_20260724_113834_522ms.csv",       # 11:38
  ("P05", "ARC-EX · before"):   D05 + "Modulated_autosave_20260724_103530_508ms.csv",   # 10:35
  ("P05", "ARC-EX · lidocaine"):D05 + "Modulated_autosave_20260724_114332_473ms.csv",   # 11:43
  ("P06", "30 Hz · before"):    D06 + "Burst_autosave_20260928_152710_593ms.csv",       # 15:27
  ("P06", "30 Hz · lidocaine"): D06 + "Burst_autosave_20260928_165756_941ms.csv",       # 16:58
  ("P06", "ARC-EX · before"):   D06 + "Modulated_autosave_20260928_154023_427ms.csv",   # 15:40
  ("P06", "ARC-EX · lidocaine"):D06 + "Modulated_autosave_20260928_170517_985ms.csv",   # 17:05
}
SUBJECTS   = ["P04", "P05", "P06"]
PROTOCOLS  = ["30 Hz", "ARC-EX"]
CONTROL    = "before"                      # the block everything is measured against
PHASES     = [CONTROL, "lidocaine"]        # the bars inside every panel
BLOCKS     = PHASES
# One panel per participant AND protocol. The threshold comes from that panel's own `before`
# recording and both of its blocks are read at it, muscle by muscle - the same rule as the
# vibration notebooks, with "before lidocaine" in place of "VIB off". 30 Hz and ARC-EX never
# share a threshold, and neither do two participants.
PANELS     = [f"{s} · {p}" for s in SUBJECTS for p in PROTOCOLS]
CONDITIONS = [f"{p} · {ph}" for p in PROTOCOLS for ph in PHASES]   # kept for section 4a

GROUP = {f"{p} · {ph}": p for p in PROTOCOLS for ph in PHASES}
BASE  = {f"{p} · {ph}": f"{p} · before" for p in PROTOCOLS for ph in PHASES}

# ======== the signal-to-noise gate ========
SNR_GATE = None    # None = draw every reading. 2.0 = leave out anything under twice its noise.
SNR_SHOW = 2.0     # still MARKS anything under this, and lists it for the noise check below
# ==========================================

# 30 Hz grey, ARC-EX red; pale = before, solid = with lidocaine
COLOUR = {"30 Hz · before": "#C6C6C6", "30 Hz · lidocaine": "#9A9A9A",
          "ARC-EX · before": "#E3A9A1", "ARC-EX · lidocaine": "#C0392B"}
PROTO_COLOUR = {"30 Hz": "#9A9A9A", "ARC-EX": "#C0392B"}

# the same four muscles as the baseline and vibration comparisons
MUSCLES = ["Flex. digitorum (L)", "Ext. digitorum (L)",      # left arm, then right
           "Flex. digitorum (R)", "Flex. carpi rad. (R)"]

KW = dict(n_pulses=10, resp_start_ms=8.0, resp_end_ms=None, guard_ms=1.0, min_snr=1.2,
          max_edge_frac=0.5, snr_on="median", anchor="mean", anchor_win_ms=3.0)

runs = {(f"{s} · {p}", ph): RUN[(s, f"{p} · {ph}")]
        for s in SUBJECTS for p in PROTOCOLS for ph in PHASES}
RAW = RUN
for pan in PANELS:
    for ph in PHASES:
        f = runs[(pan, ph)]
        amps = [m["amp_ma"] for m in load_run(f)[0]]
        print(f"{pan:16s} {ph:10s} {f.split('/')[-1][-22:]}  {amps[0]:g}-{amps[-1]:g} mA"
              + ("   <- control" if ph == CONTROL else ""))

## 2 · The motor thresholds you chose

The threshold is picked in each panel's **`before lidocaine`** recording and used for **both** of
that panel's blocks, muscle by muscle. A panel is one participant and one protocol, so
`P04 · 30 Hz` and `P04 · ARC-EX` carry their own thresholds and nothing crosses between them.

Two grids: **what is saved for each recording**, read straight from `results/<session>/mt_*.csv`,
and below the numbers **what the analysis uses** — one marked sweep per panel, that panel's
`before` threshold, drawn on its `lidocaine` row as well. That is the trace every number in
section 4b onward comes from.

The `lidocaine` picks are not used for any intensity. They are the first result in their own
right — how far the threshold moved — and section 4a reads them.

In [ ]:
CHECK = {f"{pan} · {ph}": runs[(pan, ph)] for pan in PANELS for ph in PHASES}

picked = fig_threshold_grid(CHECK, MUSCLES, title="what is saved for each recording")

### The numbers, and where each one came from

In [ ]:
# the control recording of each panel sets the intensity for both of its blocks
MT = {pan: thresholds_for(runs[(pan, CONTROL)], MUSCLES, consecutive=2, **KW) for pan in PANELS}
for pan in PANELS:
    print(f"{pan:16s} {CONTROL:10s} {threshold_source(runs[(pan, CONTROL)])}")

MT_ALL = {lab: picked.get(lab, {}) for lab in CHECK}      # every recording's own picks

# the same grid with the line on the sweep each block is analysed at
USED = {f"{pan} · {ph}": MT[pan] for pan in PANELS for ph in PHASES}
_ = fig_threshold_grid(CHECK, MUSCLES, thresholds=USED, share_gain=True,
                       title=f"the motor threshold used: each panel's {CONTROL}")

print("\nmotor threshold (mA), as picked in each recording")
print(f"{'recording':26s}" + "".join(f"{m.split(' (')[0][:8] + m[-3:]:>16s}" for m in MUSCLES))
for lab in CHECK:
    print(f"{lab:26s}" + "".join(f"{str(MT_ALL[lab].get(m) or '-'):>16s}" for m in MUSCLES))

MT_STAMP = mt_stamp(runs)

#### The two blocks on one panel

The grids above put the blocks in separate rows, which makes a difference in size hard to judge.
Here they are **drawn on top of each other**, one panel per muscle, every intensity stacked at its
own mA and **both sweeps on one scale** — so a trace that sits higher really is bigger.

Grey is `{CONTROL}`, colour is `lidocaine`. A deflection that rides on a dashed stimulus line is
artifact; one that follows it is the muscle.

In [ ]:
for pan in PANELS:
    recs, chans = [], None
    for ph in PHASES:
        if (pan, ph) not in runs:
            continue
        meta_, t_, sig_ = load_run(runs[(pan, ph)])
        recs.append((meta_, t_, sig_))
        if chans is None:
            chans = [c for m in MUSCLES for c in sig_ if pretty(c) == m]
    if len(recs) < 2:
        continue
    waterfall_overlay(recs, muscles=chans,
                      labels=[ph for ph in PHASES if (pan, ph) in runs],
                      row_h=0.34, ncol=2, colours=["#8C8C8C", "#C0392B"], xlim=(-20, 130),
                      title=f"{pan} — every intensity, both blocks on one scale")

## 3 · Are the peaks detected correctly? — every recording, every muscle

The same grid as section 2 — a row per recording, a column per muscle — but showing how the peaks
are found at the threshold picked in **that** recording. The 10 pulses are re-aligned on their own
onset, with the max (v) and min (^) the peak-to-peak is made of.

| | |
|---|---|
| **green** | the response window — `RESP_START_MS` after the pulse to `GUARD_MS` before the next |
| **red bands** | the only places a peak may be taken: `ANCHOR_WIN_MS` either side of the train's own average latency. **A peak outside these is skipped however clear it looks** |
| markers stacked | the same deflection found on every pulse |
| red ring | flagged — on a window border, or at a different latency from the rest |
| corner | `clean`, or how many of the 10 pulses were flagged |

A lidocaine block whose train sits in the noise is a *result*, not a detection failure — check
here which of the two it is before believing a small bar in section 4.

### If a peak you can see is not being taken

`TRY` changes the settings **for this figure only**, so you can test whether detection is too
strict without touching any result. When you find settings you want, copy them into `KW` in
section 1.

| setting | now | loosen it by | what it controls |
|---|---|---|---|
| `anchor_win_ms` | 3.0 | **raising** it, or `anchor=None` | how far from the train's average latency a peak may sit — much the most common reason a visible response is missed |
| `resp_start_ms` | 8.0 | **lowering** it | when the window opens after the pulse; too low and it takes the stimulus artifact |
| `min_snr` | 1.2 | **lowering** it, or `None` | how far above that muscle's noise a train must be to count at all |
| `max_edge_frac` | 0.5 | **raising** it, or `None` | how many pulses may peak on a window border before the whole train is called artifact |

In [ ]:
# ======== try different detection settings, for this figure only ========
TRY = dict(KW)                     # start from the analysis settings...
# TRY["anchor_win_ms"] = 6.0       # ...and loosen one at a time: widen the red bands
# TRY["anchor"] = None             # no anchoring: biggest max/min anywhere in the window
# TRY["resp_start_ms"] = 6.0       # open the window sooner after the pulse
# TRY["min_snr"] = None            # stop discarding trains that sit close to noise
# TRY["max_edge_frac"] = None      # stop discarding trains that peak on a window border
# ========================================================================

# at the intensity section 4 measures: every block at its panel's `before` threshold
DET_MT = {f"{pan} · {ph}": MT[pan] for pan in PANELS for ph in PHASES}
fig_detection_grid(CHECK, DET_MT, MUSCLES, edge_ms=1.0, jitter_ms=0.5,
                   title="peak detection at each recording's own threshold", **TRY)

## 4 · The results

### 4a · How far lidocaine moved the threshold

The threshold you picked **with lidocaine** as a % of the one you picked **before**, in the same
muscle, same protocol, same electrode. **100 % = the skin block cost that protocol nothing.**
Above 100 % means it needed more current once cutaneous afferents were blocked — and the
comparison that matters is **grey against red in the same muscle**: if ARC-EX rises more than
30 Hz does, more of what ARC-EX was recruiting came through the skin.

The raw milliamps are printed underneath, because a % of a 35 mA threshold and a % of a 105 mA
one are not the same thing in current.

In [ ]:
shift = {}
for pan in PANELS:
    b = MT_ALL[f"{pan} · {CONTROL}"]
    a = MT_ALL[f"{pan} · lidocaine"]
    for m in MUSCLES:
        if b.get(m) and a.get(m):
            shift[(pan, "lidocaine", m)] = dict(pct=100 * a[m] / b[m], before=b[m], after=a[m])

fig_vibration(shift, PANELS, MUSCLES, ["lidocaine"],
              colours={"lidocaine": "#C0392B"}, ylabel="threshold, % of before", ref=100,
              ylim=(0, 200), titles={pan: pan for pan in PANELS},
              title="How far lidocaine moved the motor threshold")

print(f"{'panel':16s}{'muscle':22s}{'before':>9s}{'lido':>9s}{'%':>7s}")
for pan in PANELS:
    for m in MUSCLES:
        v = shift.get((pan, "lidocaine", m))
        print(f"{pan:16s}{m:22s}"
              + (f"{v['before']:>7g} mA{v['after']:>7g} mA{v['pct']:>6.0f} %" if v
                 else f"{'-':>9s}{'-':>9s}{'-':>7s}"))

### 4b · The response at a matched intensity

Each protocol read at the intensity its **before** block defines, so the only thing that changes
between the two bars is the lidocaine. **100 % = the response did not change.** Grey is 30 Hz, red
is ARC-EX; the pale bars are the *before* blocks and sit at 100 % by construction — they are there
so the pair can be seen as a pair.

Read it against 4a: a protocol can lose response *and* need more current, or only one of the two.

In [ ]:
warn_if_stale(runs, MT_STAMP, used=[(pan, CONTROL) for pan in PANELS])

tab = condition_table(runs, MT, MUSCLES, PHASES, base=CONTROL, match="common", **KW)

# one figure: the control at 100 %, then lidocaine measured two ways
BARS = [CONTROL, "lidocaine · 1st pulse", "lidocaine · mean 1-10"]
BAR_COLOUR = {CONTROL: "#8C8C8C",
              "lidocaine · 1st pulse": "#E3A9A1", "lidocaine · mean 1-10": "#C0392B"}
both = {}
for pan in PANELS:
    for m in MUSCLES:
        if (pan, CONTROL, m) in tab:
            both[(pan, CONTROL, m)] = dict(tab[(pan, CONTROL, m)], pct=100.0)
        v = tab.get((pan, "lidocaine", m))
        if v:
            both[(pan, "lidocaine · 1st pulse", m)] = dict(v, pct=v.get("pct"))
            both[(pan, "lidocaine · mean 1-10", m)] = dict(v, pct=v.get("pct_all"))

fig_vibration(both, PANELS, MUSCLES, BARS, colours=BAR_COLOUR,
              min_snr_ratio=SNR_GATE, zero_if_absent=True,
              ylabel=f"% of {CONTROL}", ylim=(0, 200),
              titles={pan: pan for pan in PANELS},
              title=f"Lidocaine as % of {CONTROL}: 1st pulse and the whole train",
              save=None)

### The numbers, so they can be checked

In [ ]:
print(f"{'panel':16s}{'muscle':22s}{'mA':>5s}"
      + "".join(f"{ph[:10] + ' 10p':>16s}{ph[:10] + ' 1st':>16s}" for ph in PHASES))
for pan in PANELS:
    for m in MUSCLES:
        amp = next((tab[(pan, ph, m)]["amp"] for ph in PHASES if (pan, ph, m) in tab), None)
        if amp is None:
            continue
        cells = []
        for ph in PHASES:
            v, why = snr_verdict(tab, (pan, ph, m), CONTROL, SNR_GATE)
            if why:
                cells += [why, ""]
            elif v is None:
                cells += ["-", "-"]
            else:
                mk = " ?" if v.get("snr", 9e9) < SNR_SHOW else "  "
                cells += [f"{v['pct_all']:.0f} %" + mk, f"{v['pct']:.0f} %" + mk]
        print(f"{pan:16s}{m:22s}{amp:5g}" + "".join(f"{c_:>16s}" for c_ in cells))
print(f"\n10p = mean of all 10 pulses   1st = the 1st pulse only, both as % of {CONTROL}")
print(f"?  = under {SNR_SHOW:g}x its own noise - drawn, but check it in the noise panels below")

### 4c · One figure per muscle — before and with lidocaine, with the EMG

The 1st pulse in mV, the 2nd as a % of it, and the mean of pulses 2–10 as a % of it, with the EMG
underneath in matching colours. The mV panel is the quantity the two percentages are a percentage
**of**: read it within a participant only.

Along the train is a second, independent readout — if lidocaine removes a late, polysynaptic
component, the 2nd pulse and the 2–10 mean fall even where the 1st pulse barely moves.

In [ ]:
MT_cond = {(pan, ph): {m: tab[(pan, ph, m)]["amp"] for m in MUSCLES if (pan, ph, m) in tab}
           for pan in PANELS for ph in PHASES}

for m in MUSCLES:
    fig_bars(runs, MT_cond, m, PANELS, protocols=tuple(PHASES), steps=0,
             colours={CONTROL: "#8C8C8C", "lidocaine": "#C0392B"},
             names={ph: ph for ph in PHASES},
             linestyles={CONTROL: "-", "lidocaine": "-"},
             min_snr_ratio=SNR_GATE, rest_from=2, with_p1=True, traces=True,
             title=m, save=None, **KW)

### 4d · Across participants — mean ± SD, muscle by muscle

The figures above are one muscle at a time. This pools them: for each muscle, the **mean over the
three participants** with **SD**, and every participant as a **dot in its bar's colour**.

**Top figure — what lidocaine did.** The response with lidocaine as a % of the same muscle's
response before, both read at the same intensity. The pale bar is **before**, 100 % by
construction; grey is 30 Hz with lidocaine, red is ARC-EX with lidocaine. **100 % = the skin block
cost that protocol nothing.**

**Bottom figures — what the train did**, before and with lidocaine separately: the mean of pulses
2–10 as a % of each train's own 1st pulse. Read the pair to see whether blocking cutaneous
afferents changed how much the response falls along the burst — a reflex component removed should
show up here even where the 1st pulse barely moves.

`n` sits on each bar: a reading whose 1st pulse is under `2 x` its own noise is left out and named
underneath, since all of these divide by that pulse.

In [ ]:
for key, lab, ttl in (("pct_all", f"% of {CONTROL} (mean 1-10)",
                       f"Lidocaine — whole train, % of {CONTROL}"),
                      ("pct", f"% of {CONTROL} (1st pulse)",
                       f"Lidocaine — 1st pulse, % of {CONTROL}")):
    fig_group_summary(tab, PANELS, MUSCLES, PHASES, base={ph: CONTROL for ph in PHASES},
                      key=key, colours={CONTROL: "#8C8C8C", "lidocaine": "#C0392B"},
                      min_snr_ratio=SNR_GATE, err="sd", reference=(CONTROL, 100),
                      ylabel=lab, ref=100, ylim=(0, 200), title=ttl, save=None)

### 4e · Before against with lidocaine, one panel per protocol

The same comparison as 4d, laid out so the pair can be read in one glance: **30 Hz on the left,
ARC-EX on the right**, sharing a y axis. In each panel, for each muscle, the **pale** bar is
before lidocaine and the **solid** one is with it, mean over the three participants with SD, every
participant a dot in its bar's colour.

**First figure — how much the response falls along the train** (mean of pulses 2–10 as a % of that
train's own 1st pulse). Both bars are on the same footing here: each train is measured against
itself, so before and with lidocaine can be compared directly and the height difference is what
lidocaine did to the depression.

**Second figure — the same for the 2nd pulse alone**, 33 ms after the first.

Nothing is divided by the other block, so a muscle can appear in one bar and not the other; `n`
on each bar says how many participants went in, and a reading under 2x its own noise is left out
and named underneath.

In [ ]:
PANELS_BY_PROTOCOL = {p: [f"{s} · {p}" for s in SUBJECTS] for p in PROTOCOLS}

for key, lab in (("pct_all", "mean of all 10 pulses"), ("pct", "1st pulse")):
    fig_group_panels(tab, PANELS, MUSCLES, {p: ["lidocaine"] for p in PROTOCOLS}, key=key,
                     colours={"lidocaine": "#C0392B"}, min_snr_ratio=SNR_GATE,
                     reference=(CONTROL, 100), ylabel=f"% of {CONTROL}", ref=100, ylim=(0, 200),
                     title=f"Lidocaine as % of before — {lab}", save=None)

#### The whole train, first pulse included

The figures above take each train apart — the 1st pulse, the 2nd, the rest. This puts it back
together: the **average of all 10 pulses** with lidocaine, as a **% of the average of all 10
pulses before**, in the same muscle at the same intensity.

It is the closest thing to "how much EMG did the burst deliver in total, and did lidocaine change
it". **100 % = the burst delivered the same overall.** The pale bar is *before*, 100 % by
construction; the solid bar is *with lidocaine*.

Use it alongside 4d rather than instead of it. 4d compares **first pulses**, which is the
recruitment at the start of the train; this compares the **whole train**, which is what the muscle
actually received. A manipulation can leave the first response untouched and still change the
total, or the reverse.

In [ ]:
# kept: the whole-train view on its own, if you want it without the 1st-pulse bar beside it
fig_group_summary(tab, PANELS, MUSCLES, ["lidocaine"], base={"lidocaine": CONTROL},
                  key="pct_all", colours={"lidocaine": "#C0392B"},
                  min_snr_ratio=SNR_GATE, err="sd", reference=(CONTROL, 100),
                  ylabel=f"% of {CONTROL} (all 10 pulses)", ref=100, ylim=(0, 200),
                  title="Average of all 10 pulses, with lidocaine as % of before",
                  save=None)

### Is it noise, or is there nothing there? — decide by eye

For every reading marked `?`, the response windows beside windows of the same length from the
90 ms **before** the stimulus, on one scale. Alike means noise. A deflection at a consistent
latency that the pre-stimulus windows do not have means a response, however small the ratio.

In [ ]:
ONLY = None     # None = every marked reading; or ("P04 · ARC-EX", "lidocaine", "Flex. digitorum (L)")

flagged = [(pan, ph, m) for pan in PANELS for ph in PHASES for m in MUSCLES
           if (pan, ph, m) in tab and tab[(pan, ph, m)].get("snr", 9e9) < SNR_SHOW]
todo = [ONLY] if ONLY else flagged
print(f"{len(flagged)} reading(s) under {SNR_SHOW:g}x noise"
      + ("  (the gate is OFF - all of them are in the figures)" if SNR_GATE is None else "") + "\n")
for pan, ph, m in todo:
    fig_noise_check(runs[(pan, ph)], m, tab[(pan, ph, m)].get("amp"),
                    title=f"{pan} · {ph} · {m}", **KW)

## 5 · Change a threshold

**Six cells** — each panel's `before lidocaine` recording, the control everything from 4b onward
is measured against. Move the slider, press **Set to slider**, then **Save**, and re-run from
section 2.

The `lidocaine` picks feed **4a** (how far the threshold moved) and section 2's grid; they set no
intensity. To pick one, call `repick(("P04 · 30 Hz", "lidocaine"))` in a new cell.

In [ ]:
def repick(key):
    """Open the picker for one recording of this comparison, with a Save button.

    The green line is the DETECTOR's answer for this recording - not another block's, and not
    whatever is already saved (that is the orange trace)."""
    csv = runs[key]
    meta_, t_, sig_ = load_run(csv)
    chans = [c for c in sig_ if c != "Trigger A" and pretty(c) in MUSCLES]
    prev = load_threshold_csv(mt_file(csv), by="channel") if os.path.exists(mt_file(csv)) else None
    detected = motor_thresholds([dict(label="_", csv=csv)], [pretty(c) for c in chans],
                                consecutive=2, verbose=False, **KW)["_"]
    print(f"{key}  {csv.split('/')[-1]}  |  {[m['amp_ma'] for m in meta_]} mA"
          + ("  |  your picks reloaded" if prev else "  |  nothing picked yet"))
    return threshold_picker(meta_, t_, sig_, chans, xlim=(-20, 130), picks=prev, key=csv,
                            suggest=detected, suggest_label="detected",
                            on_save=lambda p_: save_threshold_csv(p_, chans, csv, meta=meta_))

In [ ]:
repick(('P04 · 30 Hz', 'before'))

In [ ]:
repick(('P04 · ARC-EX', 'before'))

In [ ]:
repick(('P05 · 30 Hz', 'before'))

In [ ]:
repick(('P05 · ARC-EX', 'before'))

In [ ]:
repick(('P06 · 30 Hz', 'before'))

In [ ]:
repick(('P06 · ARC-EX', 'before'))